<a href="https://colab.research.google.com/github/Pedroct06/Estudos-NN/blob/main/EstudosRedesNeurais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
import numpy as np

In [ ]:
#Criando a rede neural

In [ ]:
class LineNetwork(nn.Module):
  #inicio
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(nn.Linear(1,1))

  # Como a rede computa
  def forward(self, x):
    return self.layers(x)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.distributions.uniform as urand

In [ ]:
#Preparando os dados

In [ ]:
class AlgebraicDataset(Dataset):
  def __init__(self, f, interval, nsamples):
    X = urand.Uniform(interval[0], interval[1]).sample([nsamples])
    self.data = [(x,f(x)) for x in X]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return self.data[idx]


In [ ]:
#treinando a rede neural

In [ ]:
line = lambda x: 2*x + 3
interval = (-10,10)
train_nsamples = 1000
test_nsamples = 100

In [ ]:
train_dataset = AlgebraicDataset(line, interval, train_nsamples)
test_dataset = AlgebraicDataset(line, interval, test_nsamples)

train_dataloader = DataLoader(train_dataset, train_nsamples, shuffle=True)
test_dataloader = DataLoader(test_dataset, test_nsamples, shuffle=True)

In [ ]:
# Hiperparâmetros de otimização

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Rodando na {device}')

Rodando na cpu


In [ ]:
model = LineNetwork().to(device)

In [ ]:
#Função de perda
lossfunc = nn.MSELoss()
#Gradiente descendente estocástico
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

In [ ]:
def train(model, dataloader, loss, optimizer):
  model.train()
  cumloss = 0.0
  for X,y in dataloader:
    X = X.unsqueeze(1).float().to(device)
    y = y.unsqueeze(1).float().to(device)

    pred = model(X)
    loss = lossfunc(pred, y)

    #zera os gradientes acumulados
    optimizer.zero_grad()
    #computa os gradientes
    loss.backward()
    #anda na direção certa
    optimizer.step()

    #para obter o float
    cumloss += loss.item()

  return cumloss / len(dataloader)

In [ ]:
def test(model, dataloader, loss):
  model.eval()
  cumloss = 0.0
  with torch.no_grad():
    for X,y in dataloader:
      X = X.unsqueeze(1).float().to(device)
      y = y.unsqueeze(1).float().to(device)

      pred = model(X)
      loss = lossfunc(pred, y)
      cumloss += loss.item()

  return cumloss / len(dataloader)

In [ ]:
epochs = 301
for t in range(epochs):
  train_loss = train(model, train_dataloader, lossfunc, optimizer)
  if t % 10 == 0:
    print(f"Epoch: {t}; Train Loss {train_loss}")
test_loss = test(model, test_dataloader, lossfunc)
print(f"Test Loss: {test_loss}")

Epoch: 0; Train Loss 4.1859846115112305
Epoch: 10; Train Loss 4.093358516693115
Epoch: 20; Train Loss 4.010463237762451
Epoch: 30; Train Loss 3.936107873916626
Epoch: 40; Train Loss 3.8692450523376465
Epoch: 50; Train Loss 3.8089582920074463
Epoch: 60; Train Loss 3.7544431686401367
Epoch: 70; Train Loss 3.704996347427368
Epoch: 80; Train Loss 3.6599960327148438
Epoch: 90; Train Loss 3.618901252746582
Epoch: 100; Train Loss 3.581238269805908
Epoch: 110; Train Loss 3.5465898513793945
Epoch: 120; Train Loss 3.5145888328552246
Epoch: 130; Train Loss 3.4849178791046143
Epoch: 140; Train Loss 3.45729398727417
Epoch: 150; Train Loss 3.431471347808838
Epoch: 160; Train Loss 3.4072365760803223
Epoch: 170; Train Loss 3.384399175643921
Epoch: 180; Train Loss 3.362790584564209
Epoch: 190; Train Loss 3.342268705368042
Epoch: 200; Train Loss 3.322704792022705
Epoch: 210; Train Loss 3.3039863109588623
Epoch: 220; Train Loss 3.286015272140503
Epoch: 230; Train Loss 3.268707513809204
Epoch: 240; Train 

In [ ]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.activation import ReLU
class MultiNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(1,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,8),
        nn.ReLU(),
        nn.Linear(8,1),
        )
  def forward(self, x):
    return self.layers(x)

In [ ]:
multimodel = MultiNetwork().to(device)

In [ ]:
from math import cos
f = lambda x: cos(x/2)

In [ ]:
train_dataset = AlgebraicDataset(f, interval, train_nsamples)
test_dataset = AlgebraicDataset(f, interval, test_nsamples)

train_dataloader = DataLoader(train_dataset, train_nsamples, shuffle = True)
test_dataloader = DataLoader(test_dataset, test_nsamples, shuffle = True)

In [ ]:
lossFunc = nn.MSELoss()
optimizer = torch.optim.Adam(multimodel.parameters(), lr = 0.01)
epochs = 20000
for t in range(epochs):
  train_loss = train(multimodel, train_dataloader, lossFunc, optimizer)
  if t % 100 == 0:
    print(f"Epoch: {t}; Train Loss: {train_loss}")

test_loss = test(multimodel, test_dataloader, lossFunc)
print(f"Test Loss: {test_loss}")

Epoch: 0; Train Loss: 0.46152541041374207
Epoch: 100; Train Loss: 0.003030822379514575
Epoch: 200; Train Loss: 0.0009190567652694881
Epoch: 300; Train Loss: 0.0008994993404485285
Epoch: 400; Train Loss: 0.000454875233117491
Epoch: 500; Train Loss: 0.00034602556843310595
Epoch: 600; Train Loss: 0.00025462836492806673
Epoch: 700; Train Loss: 0.00023848268028814346
Epoch: 800; Train Loss: 0.0004839120665565133
Epoch: 900; Train Loss: 0.003445760812610388
Epoch: 1000; Train Loss: 0.00042359938379377127
Epoch: 1100; Train Loss: 0.0002140245196642354
Epoch: 1200; Train Loss: 0.0001498166675446555
Epoch: 1300; Train Loss: 8.938753308029845e-05
Epoch: 1400; Train Loss: 0.00022188315051607788
Epoch: 1500; Train Loss: 2.9737291697529145e-05
Epoch: 1600; Train Loss: 2.3062299078446813e-05
Epoch: 1700; Train Loss: 0.0008864289266057312
Epoch: 1800; Train Loss: 3.846465187962167e-05
Epoch: 1900; Train Loss: 0.00037401789450086653
Epoch: 2000; Train Loss: 0.00010868092795135453
Epoch: 2100; Train Lo